# 20.3 同一張照片的不同問題，為什麼要留在同一份考卷？


你用一張公園照片教模型「有人騎車」，再用同一張圖問「腳踏車在哪裡」，這是另一個問題，卻不是另一張新照片。如果前者放訓練、後者放測試，模型可能已認熟那張畫面。自然資料的第一步因此是辨認同一素材有哪些衍生題，再把它們整組分開。

前置是[5.12資料家族](https://birdhackor.github.io/tiny-perceptron-vlm/5.12.html)與[19.3先切家族再生成變體](https://birdhackor.github.io/tiny-perceptron-vlm/19.3.html)。家族在這裡以原始素材為起點：同照片的多個問題、同文件的不同裁切，都一起分配。錄音則還要考慮說話者或來源節目；只把同一段聲音切成前後兩半，不會自動得到彼此獨立的新說話者測試。

先看四張人工資料卡：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
records = [
    {"family": "公園照片A", "split": "train", "question": "人正在做什麼？"},
    {"family": "公園照片A", "split": "train", "question": "圖裡有腳踏車嗎？"},
    {"family": "招牌照片B", "split": "test", "question": "請讀出文字。"},
    {"family": "招牌照片B", "split": "test", "question": "圖片裡有中文字嗎？"},
]
families = {name: {row["family"] for row in records if row["split"] == name} for name in ["train", "test"]}
print("訓練家族", sorted(families["train"]))
print("測試家族", sorted(families["test"]))
print("跨份重疊", sorted(families["train"] & families["test"]))

每筆`row`保存原素材家族、用途與問題。內層集合把同一側重複的家族合成一項；`&`取共同項，最後應印空清單。這只檢查我們人工寫的四張卡，沒有下載照片，也沒核對真正媒體內容。正式資料還要保存來源ID、素材指紋、標註與衍生規則；家族名稱不同但圖片bytes完全相同，也仍可能交叉。

來源與標註要一起讀。例如圖片描述資料可以教「畫面中有什麼」，未必附有中文字逐字稿；語音轉寫資料可以教辨識說話內容，未必提供適合的聊天回應。把所有答案都改名`answer`，不會使這些任務等同。資料包也應保留授權與署名：資料卡的授權文字、照片本身的權利和可否重散佈，要依實際來源確認。

正式流程先留出素材，再用訓練資料更新，用驗證資料選設定。決定版本後才開最後考卷。若已看過最後錯題，再依那些題目改配方，應另建立新實驗的驗證與最後檢查，而不是繼續把原分數稱成盲測。已預訓練底座可能見過上游公開素材，來源家族互不交叉也不能替我們證明整個上游預訓練從未見過它；報告會把本課切分與這項限制分開。

本課實際整理的材料如下。表中先數原素材，再說明一份素材會產生幾道題，避免把同圖多問誤認成多張新照片。

| 材料 | 訓練 | 驗證 | 最後測試 | 用途 |
| --- | ---: | ---: | ---: | --- |
| DOCCI自然照片 | 120張 | 12張 | 12張 | 訓練每圖一題；驗證與測試每圖一份描述、兩道短問答 |
| 本課合成中文字卡 | 108張 | 10張 | 21張 | 分別產生120、12、24題，包含有字／無字、抄寫與行序 |
| FLEURS真人華語錄音 | 24段 | 6段 | 12段 | 保留原始錄音與逐字稿，另測辨識及文字／語音兩路聊天 |

自然照片採DOCCI的CC BY 4.0素材；合成字卡使用有OFL授權的Noto中文字體；真人錄音採FLEURS的CC BY 4.0素材。照片題與字卡題加上本課自寫的文字對話後，共有272道訓練題、52道驗證題、66道最後測試題。訓練文字題中的10個短例各出現三次，用來增加原本聊天行為的練習比重，不能說成30個不同例子。錄音另外按24／6／12段保存，不混入這次LoRA更新。

DOCCI原始描述是英文，訓練取原始人工描述的開頭片段並保留英文。切句程式遇到句尾引號時可能帶入下一句，因此不把每筆都稱為只有一句。留出照片的中文問題與參考答案，則由本課AI助理逐張查看照片、對照英文描述後編寫，不是DOCCI官方的人工中文標註。題目在這輪推論前已固定。

驗證與最後測試各用不同照片、不同字卡家族；同圖的各種問題留在同一側。FLEURS沿用來源的訓練、開發、測試切分，但選取的片段沒有公開個別說話者ID，不能因此自行宣稱驗證與測試的每位說話者都不同。這些是朗讀句子，並非即興聊天，也沒有證明涵蓋臺灣口音。可從[資料說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/DATA.md)與[固定資料清單](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/manifest.json)回查來源、素材指紋、授權和每道題；本課切分不重疊，仍不代表上游底座從未見過公開照片。

練習只把第二筆`split`改成`test`。先預測重疊清單會出現公園照片A，再執行核對。問題文字仍不同，檢查卻能指出同一素材跨份；修正方式是整個家族一起分配，不能只替第二筆換一個家族名字。
